# VehiAlpes — Entregable 3: consultas SQL

MINE-4214 · Modelado y Diseño de Datos · Taller 1

## Qué contiene este notebook

Dos requerimientos de negocio resueltos sobre el modelo dimensional de la capa
gold. Para cada uno se documenta el **rol del usuario** que necesita la
respuesta, la **consulta**, la **lectura del resultado** y la **decisión** que
habilita.

| # | Requerimiento | Rol del usuario | Decisión |
|---|---|---|---|
| 1 | Qué vehículos conviene comprar | Director de Compras y Flota | Reasignar el presupuesto entre gamas |
| 2 | Cuánto tiempo conservarlos en alquiler | Gerente de Operaciones de Flota | Fijar la ventana de rotación a venta |

Las dos preguntas son textualmente las que plantea el enunciado: *qué carros
comprar* y *cuáles de los vehículos prefieren dejarlos como alquiler y por
cuánto tiempo*.

> **Catálogo:** las consultas están calificadas con `vehialpes.gold`. Si tu
> catálogo tiene otro nombre, ejecuta la celda siguiente para fijar el contexto
> y las consultas funcionarán igual.

In [0]:
%sql
-- Fija el contexto. Cambia el nombre del catálogo si el tuyo es distinto.
USE CATALOG vehialpes;
USE SCHEMA gold;

---

# Requerimiento 1 — ¿Qué vehículos conviene comprar?

## Rol del usuario

**Director de Compras y Flota.** Define el plan anual de adquisiciones y negocia
con los proveedores el mix de marcas y gamas que ingresan a la flota.

## La pregunta, bien formulada

La pregunta **no** es qué marca genera más ingreso absoluto. Formulada así, la
gama alta gana siempre por tener tarifas más altas, y la conclusión sería la
contraria a la correcta. La pregunta útil es **qué proporción de su costo de
adquisición recupera cada marca**.

## Por qué este modelo es necesario para responderla

- **Cruza dos granos distintos.** La inversión y el ingreso acumulado viven en
  `hecho_ciclo_vida` (una fila por placa); la ocupación vive en
  `hecho_ocupacion_diaria` (una fila por vehículo-día). Se agregan por separado y
  se unen después: es *drill-across*. Con un join directo los días-carro
  multiplicarían los valores de compra y el resultado sería absurdo.
- **Separa poblaciones.** El valor residual se calcula solo sobre los 150
  vehículos ya vendidos. Sin el `CASE WHEN esta_vendido`, los 350 que siguen en
  flota entrarían con valor de venta nulo y hundirían el promedio.
- **Exige masa crítica.** El `HAVING COUNT(*) >= 10` evita concluir sobre marcas
  con dos o tres vehículos, donde un caso atípico decidiría el ranking.
- **Aísla el sesgo de la muestra.** La ocupación se restringe a 2024 porque la
  muestra tiene 3.137 alquileres en 2024 contra 1.230 en 2025 y 23 en 2026.

In [0]:
%sql
WITH inversion AS (            -- una fila por vehículo, con su configuración
  SELECT v.marca, v.modelo, v.tipo_combustible,
         cv.sk_vehiculo, cv.valor_compra, cv.valor_venta,
         cv.ingreso_total_alquileres, cv.dias_en_flota,
         cv.num_alquileres, cv.esta_vendido
  FROM vehialpes.gold.hecho_ciclo_vida cv
  JOIN vehialpes.gold.dim_vehiculo v ON cv.sk_vehiculo = v.sk_vehiculo
),
ocupacion AS (                 -- ocupación medida solo en 2024
  SELECT dv.marca, dv.modelo,
         SUM(o.esta_alquilado) AS dias_alquilados,
         COUNT(*)              AS dias_disponibles
  FROM vehialpes.gold.hecho_ocupacion_diaria o
  JOIN vehialpes.gold.dim_fecha    f  ON o.sk_fecha    = f.sk_fecha
  JOIN vehialpes.gold.dim_vehiculo dv ON o.sk_vehiculo = dv.sk_vehiculo
  WHERE f.anio = 2024
  GROUP BY dv.marca, dv.modelo
)
SELECT i.marca,
       COUNT(*)                                        AS vehiculos,
       ROUND(AVG(i.valor_compra) / 1e6, 1)             AS compra_prom_mcop,
       ROUND(AVG(i.ingreso_total_alquileres) / 1e6, 1) AS alquiler_prom_mcop,
       ROUND(100.0 * SUM(i.ingreso_total_alquileres)
                   / SUM(i.valor_compra), 1)           AS pct_recuperado_alquiler,
       ROUND(100.0 * SUM(o.dias_alquilados)
                   / SUM(o.dias_disponibles), 1)       AS pct_ocupacion_2024,
       ROUND(AVG(CASE WHEN i.esta_vendido
                      THEN 100.0 * i.valor_venta / i.valor_compra END), 1)
                                                       AS pct_recuperado_venta,
       ROUND(AVG(i.ingreso_total_alquileres
                 / NULLIF(i.dias_en_flota, 0)) / 1000, 1) AS ingreso_dia_kcop
FROM inversion i
JOIN ocupacion o ON i.marca = o.marca AND i.modelo = o.modelo
GROUP BY i.marca
HAVING COUNT(*) >= 10
ORDER BY pct_recuperado_alquiler DESC;

marca,vehiculos,compra_prom_mcop,alquiler_prom_mcop,pct_recuperado_alquiler,pct_ocupacion_2024,pct_recuperado_venta,ingreso_dia_kcop
Suzuki,18,90.6,17.5,19.3,14.1,67.9,18.8
Porsche,110,271.1,31.2,11.5,16.2,68.4,32.3
Alfa Romeo,33,255.7,26.7,10.4,14.9,65.8,30.2
Mercedes-Benz,83,276.1,25.7,9.3,15.3,63.9,28.0
BMW,157,272.7,19.9,7.3,14.6,64.6,23.4
Audi,62,289.3,18.5,6.4,15.3,61.5,20.5
Volvo,14,295.2,15.0,5.1,14.7,65.2,15.9


## Lectura del resultado

| Marca | Vehículos | Compra prom. (M) | Recuperado vía alquiler | Ocupación 2024 | Recuperado vía venta | Ingreso/día (K) |
|---|---|---|---|---|---|---|
| Suzuki | 18 | 90,6 | **19,3%** | 14,1% | 67,9% | 18,8 |
| Porsche | 110 | 271,1 | 11,5% | 16,2% | 68,4% | 32,3 |
| Alfa Romeo | 33 | 255,7 | 10,4% | 14,9% | 65,8% | 30,2 |
| Mercedes-Benz | 83 | 276,1 | 9,3% | 15,3% | 63,9% | 28,0 |
| BMW | 157 | 272,7 | 7,3% | 14,6% | 64,6% | 23,4 |
| Audi | 62 | 289,3 | 6,4% | 15,3% | 61,5% | 20,5 |
| Volvo | 14 | 295,2 | 5,1% | 14,7% | 65,2% | 15,9 |

**El hallazgo central: la ocupación es prácticamente idéntica en todas las
marcas**, entre 14,1% y 16,2%. Los clientes alquilan lo que está disponible, no
eligen por marca.

Pero la tarifa diaria **no escala en proporción al precio de compra**. Un Suzuki
de 90 millones genera 18.800 pesos por día en flota; un Porsche de 271 millones
genera 32.300. Tres veces la inversión para menos del doble del ingreso.

El valor residual tampoco compensa: está entre 61,5% y 68,4% en todas las
marcas, sin relación con el precio de compra.

## Decisión que habilita

Suzuki recupera vía alquiler **tres veces más** que Volvo y casi el triple que
Audi, con la misma ocupación y un valor residual comparable.

- **Aumentar** la participación de gama media en el próximo plan de compras.
- **Reducir** Audi y Volvo: son las dos más caras de adquirir y las de peor
  retorno en ambas vías, alquiler y reventa.
- **Validar antes de escalar:** la conclusión sobre Suzuki descansa en 18
  vehículos. La acción responsable es una compra piloto ampliada, no un giro
  completo del mix.

---

# Requerimiento 2 — ¿Cuánto tiempo conservar un vehículo en alquiler?

## Rol del usuario

**Gerente de Operaciones de Flota.** Decide, mes a mes, qué vehículos pasan de la
flota de alquiler al inventario de venta de usados.

## La pregunta

Existe un punto en el que retener un vehículo deja de convenir: su ingreso por
alquiler ya no compensa la pérdida de valor de reventa. ¿Dónde está ese punto?

## Por qué este modelo es necesario para responderla

- **Reexpresa el tiempo.** La consulta convierte tiempo calendario en
  *antigüedad del vehículo* usando `dias_desde_ingreso`. Ningún hecho
  transaccional tiene esa columna: un contrato de alquiler sabe cuándo ocurrió,
  no cuántos meses llevaba el carro en flota. **Sin el snapshot periódico esta
  curva no existe.**
- **Normaliza por exposición.** Cada tramo de antigüedad tiene distinto número de
  vehículos y de días. Expresar todo como porcentaje sobre `dias_expuestos` es lo
  que hace comparables los tramos, y es el tratamiento correcto de una medida
  semi-aditiva: se suma dentro de un grupo de días, nunca a lo largo del tiempo.
- **Cruza ingreso con depreciación.** Una consulta que mirara solo el ingreso
  concluiría que conviene retener indefinidamente. El valor residual es lo que
  revela el costo de esperar.

In [0]:
%sql
WITH dias_con_edad AS (        -- cada día-carro etiquetado con su antigüedad
  SELECT o.sk_vehiculo,
         FLOOR(o.dias_desde_ingreso / 90) AS trimestre_en_flota,
         o.esta_alquilado,
         o.tarifa_vigente_dia
  FROM vehialpes.gold.hecho_ocupacion_diaria o
  JOIN vehialpes.gold.dim_fecha f ON o.sk_fecha = f.sk_fecha
  WHERE f.anio = 2024          -- aísla el sesgo temporal de la muestra
),
por_edad AS (
  SELECT trimestre_en_flota,
         COUNT(DISTINCT sk_vehiculo)              AS vehiculos,
         SUM(esta_alquilado)                      AS dias_alquilados,
         COUNT(*)                                 AS dias_expuestos,
         SUM(esta_alquilado * tarifa_vigente_dia) AS ingreso_teorico
  FROM dias_con_edad
  GROUP BY trimestre_en_flota
),
reventa AS (                   -- valor residual según antigüedad al vender
  SELECT FLOOR(cv.dias_en_flota / 90) AS trimestre_en_flota,
         AVG(100.0 * cv.valor_venta / cv.valor_compra) AS pct_valor_residual,
         COUNT(*) AS ventas
  FROM vehialpes.gold.hecho_ciclo_vida cv
  WHERE cv.esta_vendido        -- excluye los 350 en flota: evita sesgo de
  GROUP BY FLOOR(cv.dias_en_flota / 90)          -- supervivencia
)
SELECT e.trimestre_en_flota                                   AS trim,
       CONCAT(CAST(e.trimestre_en_flota * 3 AS STRING), '-',
              CAST(e.trimestre_en_flota * 3 + 3 AS STRING),
              ' meses')                                       AS antiguedad,
       e.vehiculos,
       e.dias_expuestos,
       ROUND(100.0 * e.dias_alquilados / e.dias_expuestos, 1) AS pct_ocupacion,
       ROUND(e.ingreso_teorico / NULLIF(e.dias_expuestos, 0) / 1000, 1)
                                                              AS ingreso_dia_kcop,
       ROUND(r.pct_valor_residual, 1)                         AS pct_valor_residual,
       r.ventas
FROM por_edad e
LEFT JOIN reventa r ON e.trimestre_en_flota = r.trimestre_en_flota
WHERE e.dias_expuestos >= 500  -- umbral mínimo de exposición por tramo
ORDER BY e.trimestre_en_flota;

trim,antiguedad,vehiculos,dias_expuestos,pct_ocupacion,ingreso_dia_kcop,pct_valor_residual,ventas
0,0-3 meses,248,17117,4.5,15.2,null,null
1,3-6 meses,396,27132,11.2,37.6,null,null
2,6-9 meses,570,35165,15.0,52.3,54.2,1
3,9-12 meses,663,36387,17.6,55.1,64.2,3
4,12-15 meses,544,28481,20.2,58.0,70.7,6
5,15-18 meses,408,18475,18.7,48.1,64.8,18
6,18-21 meses,229,9051,17.7,41.4,64.0,30
7,21-24 meses,73,2282,12.0,29.6,68.7,24


## Lectura del resultado

| Antigüedad | Vehículos | Ocupación | Ingreso/día (K) | Valor residual | Ventas |
|---|---|---|---|---|---|
| 0-3 meses | 248 | 4,5% | 15,2 | — | — |
| 3-6 meses | 396 | 11,2% | 37,6 | — | — |
| 6-9 meses | 570 | 15,0% | 52,3 | 54,2% | 1 |
| 9-12 meses | 663 | 17,6% | 55,1 | 64,2% | 3 |
| **12-15 meses** | 544 | **20,2%** | **58,0** | **70,7%** | 6 |
| 15-18 meses | 408 | 18,7% | 48,1 | 64,8% | 18 |
| 18-21 meses | 229 | 17,7% | 41,4 | 64,0% | 30 |
| 21-24 meses | 73 | 12,0% | 29,6 | 68,7% | 24 |

Las tres curvas alcanzan su máximo en el **mismo tramo, 12-15 meses**: ocupación
20,2%, ingreso diario 58.000 y valor residual 70,7%.

Después de ese punto las tres caen a la vez, que es el peor escenario posible:
el vehículo **se alquila menos y además vale menos**. Entre 12-15 y 21-24 meses
el ingreso diario cae casi a la mitad, de 58.000 a 29.600.

## Decisión que habilita

**Fijar la ventana de rotación en 15 meses.** Pasado ese punto se pierde por los
dos lados simultáneamente.

Es una regla aplicable de inmediato a los 350 vehículos que siguen en flota: los
que superen 15 meses son candidatos a pasar a inventario de usados.

Y un dato que refuerza la política: las ventas observadas se concentran entre 15
y 24 meses (72 de las 150). Es decir, **VehiAlpes ya vende tarde**, justo
después del punto óptimo. La política no propone un cambio de dirección, propone
adelantar lo que ya hace.